# Constrained RL for Aggressive Aircraft Maneuvers

This notebook trains a reinforcement learning agent to perform aggressive aircraft maneuvers (0° to 180° heading change) while respecting hard safety constraints on:
- Load factors (-3g to +7g)
- Velocity (100-300 m/s)
- Angle of attack, rates, and attitudes

**Algorithm**: PPO with Lagrangian constraint handling

**Runtime**: ~30-60 minutes on Colab GPU

## 1. Setup and Installation

In [1]:
# Install required packages
!pip install -q gymnasium torch numpy scipy matplotlib

In [2]:
# Import libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
from scipy.integrate import odeint
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from collections import deque
from IPython.display import clear_output
import time

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

✓ All imports successful
PyTorch version: 2.10.0+cu128
GPU available: True


## 2. Aircraft 6-DOF Dynamics Model

In [3]:
class Aircraft6DOF:
    """
    6-DOF aircraft dynamics model with aerodynamics
    State: [u, v, w, p, q, r, phi, theta, psi, x, y, z]
    """

    def __init__(self):
        # F-16 inspired parameters
        self.mass = 9295.44  # kg
        self.g = 9.81  # m/s^2

        # Moments of inertia (kg*m^2)
        self.Ixx = 12875
        self.Iyy = 75674
        self.Izz = 85552
        self.Ixz = 1331

        # Aerodynamic reference parameters
        self.S = 27.87  # wing area (m^2)
        self.b = 9.144  # wingspan (m)
        self.c_bar = 3.45  # mean aerodynamic chord (m)
        self.rho = 1.225  # air density (kg/m^3)

        # Aerodynamic coefficients
        self.CL0 = 0.2
        self.CLalpha = 3.45
        self.CLq = 1.7
        self.CLde = 0.36

        self.CD0 = 0.025
        self.CDalpha = 0.3
        self.CDalpha2 = 2.0

        self.CYbeta = -0.98
        self.CYp = 0.0
        self.CYr = 0.0
        self.CYda = 0.0
        self.CYdr = 0.17

        self.Clbeta = -0.12
        self.Clp = -0.26
        self.Clr = 0.14
        self.Clda = 0.13
        self.Cldr = 0.0105

        self.Cm0 = 0.0
        self.Cmalpha = -0.38
        self.Cmq = -3.6
        self.Cmde = -0.5

        self.Cnbeta = 0.25
        self.Cnp = 0.022
        self.Cnr = -0.35
        self.Cnda = 0.0012
        self.Cndr = -0.032

        self.max_thrust = 75000  # N

    def get_aerodynamic_coefficients(self, state, controls):
        """Calculate aerodynamic coefficients"""
        u, v, w = state[0:3]
        p, q, r = state[3:6]

        delta_a, delta_e, delta_r, throttle = controls

        V = np.sqrt(u**2 + v**2 + w**2)
        if V < 1.0:
            V = 1.0

        alpha = np.arctan2(w, u)
        beta = np.arcsin(v / V) if V > 0 else 0

        p_hat = p * self.b / (2 * V)
        q_hat = q * self.c_bar / (2 * V)
        r_hat = r * self.b / (2 * V)

        CL = self.CL0 + self.CLalpha * alpha + self.CLq * q_hat + self.CLde * delta_e
        CD = self.CD0 + self.CDalpha * abs(alpha) + self.CDalpha2 * alpha**2
        CY = self.CYbeta * beta + self.CYp * p_hat + self.CYr * r_hat + self.CYda * delta_a + self.CYdr * delta_r
        Cl = self.Clbeta * beta + self.Clp * p_hat + self.Clr * r_hat + self.Clda * delta_a + self.Cldr * delta_r
        Cm = self.Cm0 + self.Cmalpha * alpha + self.Cmq * q_hat + self.Cmde * delta_e
        Cn = self.Cnbeta * beta + self.Cnp * p_hat + self.Cnr * r_hat + self.Cnda * delta_a + self.Cndr * delta_r

        return CL, CD, CY, Cl, Cm, Cn, alpha, beta, V

    def dynamics(self, state, controls):
        """6-DOF equations of motion"""
        u, v, w, p, q, r, phi, theta, psi, x, y, z = state
        delta_a, delta_e, delta_r, throttle = controls

        CL, CD, CY, Cl, Cm, Cn, alpha, beta, V = self.get_aerodynamic_coefficients(state, controls)

        Q = 0.5 * self.rho * V**2

        D = Q * self.S * CD
        L = Q * self.S * CL
        Y = Q * self.S * CY

        cos_alpha = np.cos(alpha)
        sin_alpha = np.sin(alpha)

        Fx_aero = -D * cos_alpha + L * sin_alpha
        Fy_aero = Y
        Fz_aero = -D * sin_alpha - L * cos_alpha

        Fx_thrust = self.max_thrust * throttle

        Fx = Fx_aero + Fx_thrust
        Fy = Fy_aero
        Fz = Fz_aero

        Mx = Q * self.S * self.b * Cl
        My = Q * self.S * self.c_bar * Cm
        Mz = Q * self.S * self.b * Cn

        sin_phi = np.sin(phi)
        cos_phi = np.cos(phi)
        sin_theta = np.sin(theta)
        cos_theta = np.cos(theta)
        sin_psi = np.sin(psi)
        cos_psi = np.cos(psi)

        gx = -self.mass * self.g * sin_theta
        gy = self.mass * self.g * cos_theta * sin_phi
        gz = self.mass * self.g * cos_theta * cos_phi

        u_dot = (Fx + gx) / self.mass + r * v - q * w
        v_dot = (Fy + gy) / self.mass - r * u + p * w
        w_dot = (Fz + gz) / self.mass + q * u - p * v

        gamma = self.Ixx * self.Izz - self.Ixz**2

        p_dot = (self.Izz * Mx + self.Ixz * Mz +
                 self.Ixz * (self.Ixx - self.Iyy + self.Izz) * p * q -
                 (self.Izz * (self.Izz - self.Iyy) + self.Ixz**2) * q * r) / gamma

        q_dot = (My + (self.Izz - self.Ixx) * p * r - self.Ixz * (p**2 - r**2)) / self.Iyy

        r_dot = (self.Ixz * Mx + self.Ixx * Mz +
                 (self.Ixx * (self.Ixx - self.Iyy) + self.Ixz**2) * p * q -
                 self.Ixz * (self.Ixx - self.Iyy + self.Izz) * q * r) / gamma

        tan_theta = np.tan(theta)
        sec_theta = 1.0 / np.cos(theta) if abs(np.cos(theta)) > 0.01 else 100.0

        phi_dot = p + (q * sin_phi + r * cos_phi) * tan_theta
        theta_dot = q * cos_phi - r * sin_phi
        psi_dot = (q * sin_phi + r * cos_phi) * sec_theta

        x_dot = (cos_theta * cos_psi * u +
                 (sin_phi * sin_theta * cos_psi - cos_phi * sin_psi) * v +
                 (cos_phi * sin_theta * cos_psi + sin_phi * sin_psi) * w)

        y_dot = (cos_theta * sin_psi * u +
                 (sin_phi * sin_theta * sin_psi + cos_phi * cos_psi) * v +
                 (cos_phi * sin_theta * sin_psi - sin_phi * cos_psi) * w)

        z_dot = (-sin_theta * u + sin_phi * cos_theta * v + cos_phi * cos_theta * w)

        return np.array([u_dot, v_dot, w_dot, p_dot, q_dot, r_dot,
                        phi_dot, theta_dot, psi_dot, x_dot, y_dot, z_dot])

    def step(self, state, controls, dt):
        """Integrate one timestep using RK4"""
        k1 = self.dynamics(state, controls)
        k2 = self.dynamics(state + 0.5 * dt * k1, controls)
        k3 = self.dynamics(state + 0.5 * dt * k2, controls)
        k4 = self.dynamics(state + dt * k3, controls)

        new_state = state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

        new_state[6] = np.arctan2(np.sin(new_state[6]), np.cos(new_state[6]))
        new_state[7] = np.clip(new_state[7], -np.pi/2 + 0.01, np.pi/2 - 0.01)
        new_state[8] = np.arctan2(np.sin(new_state[8]), np.cos(new_state[8]))

        return new_state

    def get_load_factor(self, state, controls):
        """Calculate load factor (g's)"""
        u, v, w = state[0:3]
        phi, theta = state[6:8]

        _, _, _, _, _, _, _, _, V = self.get_aerodynamic_coefficients(state, controls)
        CL, CD, CY, _, _, _, alpha, beta, _ = self.get_aerodynamic_coefficients(state, controls)
        Q = 0.5 * self.rho * V**2

        L = Q * self.S * CL
        D = Q * self.S * CD

        cos_alpha = np.cos(alpha)
        sin_alpha = np.sin(alpha)

        Fz_aero = -D * sin_alpha - L * cos_alpha
        Fz_total = Fz_aero + self.mass * self.g * np.cos(theta) * np.cos(phi)

        nz = -Fz_total / (self.mass * self.g)
        return nz

    def get_velocity_magnitude(self, state):
        """Get total velocity magnitude"""
        u, v, w = state[0:3]
        return np.sqrt(u**2 + v**2 + w**2)

print("✓ Aircraft6DOF class defined")

✓ Aircraft6DOF class defined


## 3. Constrained Environment

In [4]:
class ConstrainedAircraftEnv(gym.Env):
    """
    Constrained aircraft environment for heading change maneuver
    Task: Change heading from 0 to 180 degrees with slight altitude change
    """

    def __init__(self):
        super(ConstrainedAircraftEnv, self).__init__()

        self.aircraft = Aircraft6DOF()
        self.dt = 0.02
        self.max_steps = 1000

        self.initial_velocity = 200.0
        self.initial_altitude = 3000.0
        self.target_heading = np.pi
        self.altitude_tolerance = 50.0

        # HARD CONSTRAINTS
        self.max_load_factor = 7.0
        self.min_load_factor = -3.0
        self.max_velocity = 300.0
        self.min_velocity = 100.0
        self.max_angle_of_attack = np.deg2rad(25)
        self.min_angle_of_attack = np.deg2rad(-10)
        self.max_roll_rate = np.deg2rad(180)
        self.max_pitch_rate = np.deg2rad(60)
        self.max_yaw_rate = np.deg2rad(60)
        self.max_roll_angle = np.deg2rad(85)
        self.max_pitch_angle = np.deg2rad(60)

        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0, -1.0, 0.0]),
            high=np.array([1.0, 1.0, 1.0, 1.0]),
            dtype=np.float32
        )

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(13,),
            dtype=np.float32
        )

        self.constraint_violations = []
        self.constraint_costs = []

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        u0 = self.initial_velocity
        self.state = np.array([u0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -self.initial_altitude])
        self.step_count = 0
        self.constraint_violations = []
        self.constraint_costs = []
        self.cumulative_constraint_cost = 0.0

        return self._get_obs(), {}

    def _get_obs(self):
        """Get normalized observation"""
        u, v, w, p, q, r, phi, theta, psi, x, y, z = self.state

        V = self.aircraft.get_velocity_magnitude(self.state)
        nz = self.aircraft.get_load_factor(self.state, np.array([0, 0, 0, 0.5]))
        alpha = np.arctan2(w, u) if u > 0 else 0.0
        psi_error = self._angle_difference(psi, self.target_heading)
        altitude = -z
        altitude_error = altitude - self.initial_altitude

        obs = np.array([
            u / 250.0, v / 50.0, w / 50.0,
            p / self.max_roll_rate, q / self.max_pitch_rate, r / self.max_yaw_rate,
            phi / self.max_roll_angle, theta / self.max_pitch_angle,
            psi_error / np.pi, altitude_error / 100.0,
            V / 250.0, nz / 7.0, alpha / self.max_angle_of_attack
        ], dtype=np.float32)

        return obs

    def _angle_difference(self, angle1, angle2):
        diff = angle1 - angle2
        return np.arctan2(np.sin(diff), np.cos(diff))

    def _scale_action(self, action):
        delta_a = action[0] * np.deg2rad(21.5)
        delta_e = action[1] * np.deg2rad(25.0)
        delta_r = action[2] * np.deg2rad(30.0)
        throttle = action[3]
        return np.array([delta_a, delta_e, delta_r, throttle])

    def _check_constraints(self, controls):
        violations = {}
        constraint_cost = 0.0
        is_safe = True

        u, v, w, p, q, r, phi, theta, psi, x, y, z = self.state

        nz = self.aircraft.get_load_factor(self.state, controls)
        if nz > self.max_load_factor:
            violations['load_factor_high'] = nz - self.max_load_factor
            constraint_cost += (nz - self.max_load_factor) ** 2 * 100
            is_safe = False
        if nz < self.min_load_factor:
            violations['load_factor_low'] = self.min_load_factor - nz
            constraint_cost += (self.min_load_factor - nz) ** 2 * 100
            is_safe = False

        V = self.aircraft.get_velocity_magnitude(self.state)
        if V > self.max_velocity:
            violations['velocity_high'] = V - self.max_velocity
            constraint_cost += (V - self.max_velocity) ** 2 * 10
            is_safe = False
        if V < self.min_velocity:
            violations['velocity_low'] = self.min_velocity - V
            constraint_cost += (self.min_velocity - V) ** 2 * 10
            is_safe = False

        alpha = np.arctan2(w, u) if u > 0 else 0.0
        if abs(alpha) > self.max_angle_of_attack:
            violations['alpha'] = abs(alpha) - self.max_angle_of_attack
            constraint_cost += (abs(alpha) - self.max_angle_of_attack) ** 2 * 200
            is_safe = False

        if abs(p) > self.max_roll_rate:
            violations['roll_rate'] = abs(p) - self.max_roll_rate
            constraint_cost += (abs(p) - self.max_roll_rate) ** 2 * 50
            is_safe = False
        if abs(q) > self.max_pitch_rate:
            violations['pitch_rate'] = abs(q) - self.max_pitch_rate
            constraint_cost += (abs(q) - self.max_pitch_rate) ** 2 * 50
            is_safe = False
        if abs(r) > self.max_yaw_rate:
            violations['yaw_rate'] = abs(r) - self.max_yaw_rate
            constraint_cost += (abs(r) - self.max_yaw_rate) ** 2 * 50
            is_safe = False

        if abs(phi) > self.max_roll_angle:
            violations['roll_angle'] = abs(phi) - self.max_roll_angle
            constraint_cost += (abs(phi) - self.max_roll_angle) ** 2 * 100
            is_safe = False
        if abs(theta) > self.max_pitch_angle:
            violations['pitch_angle'] = abs(theta) - self.max_pitch_angle
            constraint_cost += (abs(theta) - self.max_pitch_angle) ** 2 * 100
            is_safe = False

        return is_safe, constraint_cost, violations

    def _apply_safety_layer(self, controls):
        safe_controls = controls.copy()
        u, v, w = self.state[0:3]
        alpha = np.arctan2(w, u) if u > 0 else 0.0

        if abs(alpha) > 0.8 * self.max_angle_of_attack:
            safe_controls[1] *= 0.5

        V = self.aircraft.get_velocity_magnitude(self.state)
        if V > 0.9 * self.max_velocity:
            safe_controls[3] = min(safe_controls[3], 0.3)
        elif V < 1.1 * self.min_velocity:
            safe_controls[3] = max(safe_controls[3], 0.7)

        return safe_controls

    def step(self, action):
        controls = self._scale_action(action)
        is_safe, constraint_cost, violations = self._check_constraints(controls)

        if not is_safe:
            controls = self._apply_safety_layer(controls)
            is_safe, constraint_cost, violations = self._check_constraints(controls)

        self.state = self.aircraft.step(self.state, controls, self.dt)
        self.step_count += 1
        self.constraint_violations.append(violations)
        self.constraint_costs.append(constraint_cost)
        self.cumulative_constraint_cost += constraint_cost

        reward = self._compute_reward(constraint_cost)
        terminated = self._check_termination()
        truncated = self.step_count >= self.max_steps

        info = {
            'constraint_cost': constraint_cost,
            'violations': violations,
            'is_safe': is_safe,
            'load_factor': self.aircraft.get_load_factor(self.state, controls),
            'velocity': self.aircraft.get_velocity_magnitude(self.state),
            'heading_error': abs(self._angle_difference(self.state[8], self.target_heading)),
            'altitude_error': abs(-self.state[11] - self.initial_altitude)
        }

        return self._get_obs(), reward, terminated, truncated, info

    def _compute_reward(self, constraint_cost):
        u, v, w, p, q, r, phi, theta, psi, x, y, z = self.state

        psi_error = abs(self._angle_difference(psi, self.target_heading))
        heading_reward = -psi_error * 10.0

        altitude = -z
        altitude_error = abs(altitude - self.initial_altitude)
        altitude_penalty = -min(altitude_error, self.altitude_tolerance) * 0.5

        V = self.aircraft.get_velocity_magnitude(self.state)
        velocity_error = abs(V - self.initial_velocity)
        velocity_penalty = -velocity_error * 0.1

        control_penalty = -0.01 * (p**2 + q**2 + r**2)
        constraint_penalty = -constraint_cost

        completion_bonus = 0.0
        if psi_error < np.deg2rad(5) and altitude_error < self.altitude_tolerance:
            completion_bonus = 100.0

        total_reward = (heading_reward + altitude_penalty + velocity_penalty +
                       control_penalty + constraint_penalty + completion_bonus)

        return total_reward

    def _check_termination(self):
        u, v, w, p, q, r, phi, theta, psi, x, y, z = self.state

        psi_error = abs(self._angle_difference(psi, self.target_heading))
        altitude_error = abs(-z - self.initial_altitude)

        if psi_error < np.deg2rad(2) and altitude_error < self.altitude_tolerance:
            return True

        V = self.aircraft.get_velocity_magnitude(self.state)
        if V < 0.8 * self.min_velocity or V > 1.2 * self.max_velocity:
            return True

        if abs(theta) > np.deg2rad(75):
            return True

        if -z < 500 or -z > 10000:
            return True

        return False

print("✓ ConstrainedAircraftEnv class defined")

✓ ConstrainedAircraftEnv class defined


## 4. PPO-Lagrangian Agent

In [5]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=256):
        super(ActorCritic, self).__init__()

        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh()
        )

        self.actor_mean = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, action_dim)
        )

        self.actor_log_std = nn.Parameter(torch.zeros(action_dim))

        self.critic = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.cost_critic = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
            nn.init.constant_(module.bias, 0.0)

    def forward(self, obs):
        features = self.shared(obs)
        action_mean = self.actor_mean(features)
        action_std = torch.exp(self.actor_log_std)
        value = self.critic(features)
        cost_value = self.cost_critic(features)
        return action_mean, action_std, value, cost_value

    def get_action(self, obs, deterministic=False):
        action_mean, action_std, value, cost_value = self.forward(obs)
        if deterministic:
            action = action_mean
        else:
            dist = Normal(action_mean, action_std)
            action = dist.sample()
        return action, value, cost_value

    def evaluate_actions(self, obs, actions):
        action_mean, action_std, value, cost_value = self.forward(obs)
        dist = Normal(action_mean, action_std)
        log_probs = dist.log_prob(actions).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return log_probs, value, cost_value, entropy


class PPOLagrangian:
    def __init__(self, env, lr=3e-4, gamma=0.99, gae_lambda=0.95,
                 clip_epsilon=0.2, c1=0.5, c2=0.01, cost_limit=0.0,
                 lagrange_lr=0.01, hidden_dim=256):

        self.env = env
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.c1 = c1
        self.c2 = c2
        self.cost_limit = cost_limit

        obs_dim = env.observation_space.shape[0]
        action_dim = env.action_space.shape[0]

        self.policy = ActorCritic(obs_dim, action_dim, hidden_dim)
        if torch.cuda.is_available():
            self.policy = self.policy.cuda()

        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.lambda_value = 0.0  # Simple float
# Update manually (no autograd needed):
        self.lambda_value = max(0.0, self.lambda_value + lr * (cost - limit))
        self.lambda_optimizer = optim.Adam([self.lambda_net], lr=lagrange_lr)

        self.buffer = {
            'obs': [], 'actions': [], 'rewards': [], 'costs': [],
            'values': [], 'cost_values': [], 'log_probs': [], 'dones': []
        }

        self.episode_rewards = deque(maxlen=100)
        self.episode_costs = deque(maxlen=100)
        self.episode_lengths = deque(maxlen=100)
        self.constraint_violations = deque(maxlen=100)

    def select_action(self, obs, deterministic=False):
        obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
        if torch.cuda.is_available():
            obs_tensor = obs_tensor.cuda()

        with torch.no_grad():
            action, value, cost_value = self.policy.get_action(obs_tensor, deterministic)
            action_mean, action_std, _, _ = self.policy(obs_tensor)
            dist = Normal(action_mean, action_std)
            log_prob = dist.log_prob(action).sum(dim=-1)

        return action.cpu().squeeze(0).numpy(), log_prob.cpu().item(), value.cpu().item(), cost_value.cpu().item()

    def store_transition(self, obs, action, reward, cost, value, cost_value, log_prob, done):
        self.buffer['obs'].append(obs)
        self.buffer['actions'].append(action)
        self.buffer['rewards'].append(reward)
        self.buffer['costs'].append(cost)
        self.buffer['values'].append(value)
        self.buffer['cost_values'].append(cost_value)
        self.buffer['log_probs'].append(log_prob)
        self.buffer['dones'].append(done)

    def compute_gae(self, rewards, values, dones, next_value):
        advantages = []
        gae = 0
        values = values + [next_value]

        for t in reversed(range(len(rewards))):
            delta = rewards[t] + self.gamma * values[t + 1] * (1 - dones[t]) - values[t]
            gae = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * gae
            advantages.insert(0, gae)

        returns = [adv + val for adv, val in zip(advantages, values[:-1])]
        return advantages, returns

    def update(self, n_epochs=10, batch_size=64):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        obs = torch.FloatTensor(np.array(self.buffer['obs'])).to(device)
        actions = torch.FloatTensor(np.array(self.buffer['actions'])).to(device)
        old_log_probs = torch.FloatTensor(np.array(self.buffer['log_probs'])).to(device)

        rewards = self.buffer['rewards']
        costs = self.buffer['costs']
        values = self.buffer['values']
        cost_values = self.buffer['cost_values']
        dones = self.buffer['dones']

        next_value = 0.0
        next_cost_value = 0.0

        reward_advantages, reward_returns = self.compute_gae(rewards, values, dones, next_value)
        cost_advantages, cost_returns = self.compute_gae(costs, cost_values, dones, next_cost_value)

        reward_advantages = torch.FloatTensor(reward_advantages).to(device)
        reward_returns = torch.FloatTensor(reward_returns).to(device)
        cost_returns = torch.FloatTensor(cost_returns).to(device)

        reward_advantages = (reward_advantages - reward_advantages.mean()) / (reward_advantages.std() + 1e-8)

        dataset_size = obs.shape[0]

        for epoch in range(n_epochs):
            indices = np.arange(dataset_size)
            np.random.shuffle(indices)

            for start_idx in range(0, dataset_size, batch_size):
                end_idx = min(start_idx + batch_size, dataset_size)
                batch_indices = indices[start_idx:end_idx]

                batch_obs = obs[batch_indices]
                batch_actions = actions[batch_indices]
                batch_old_log_probs = old_log_probs[batch_indices]
                batch_reward_advantages = reward_advantages[batch_indices]
                batch_reward_returns = reward_returns[batch_indices]
                batch_cost_returns = cost_returns[batch_indices]

                log_probs, values, cost_values, entropy = self.policy.evaluate_actions(batch_obs, batch_actions)

                ratio = torch.exp(log_probs - batch_old_log_probs)
                surr1 = ratio * batch_reward_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * batch_reward_advantages

                lambda_value = torch.clamp(self.lambda_net, min=0.0)
                cost_penalty = lambda_value * log_probs

                policy_loss = -torch.min(surr1, surr2).mean() - self.c2 * entropy.mean()
                policy_loss += cost_penalty.mean()

                value_loss = self.c1 * ((values.squeeze() - batch_reward_returns) ** 2).mean()
                cost_value_loss = self.c1 * ((cost_values.squeeze() - batch_cost_returns) ** 2).mean()

                loss = policy_loss + value_loss + cost_value_loss

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.optimizer.step()

                mean_cost = torch.FloatTensor(costs).mean()
                lambda_loss = -lambda_value * (self.cost_limit - mean_cost)

                self.lambda_optimizer.zero_grad()
                lambda_loss.backward()
                self.lambda_optimizer.step()

        for key in self.buffer:
            self.buffer[key] = []

        return {
            'policy_loss': policy_loss.item(),
            'value_loss': value_loss.item(),
            'cost_value_loss': cost_value_loss.item(),
            'lambda': torch.clamp(self.lambda_net, min=0.0).item(),
            'mean_cost': mean_cost.item()
        }

print("✓ ActorCritic and PPOLagrangian classes defined")

✓ ActorCritic and PPOLagrangian classes defined


## 5. Training Configuration

In [6]:
# Training hyperparameters
TOTAL_TIMESTEPS = 200000  # Reduced for Colab (increase to 500k for better results)
UPDATE_FREQ = 2048
LOG_FREQ = 5

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("✓ Training configuration set")
print(f"  Total timesteps: {TOTAL_TIMESTEPS:,}")
print(f"  Update frequency: {UPDATE_FREQ}")
print(f"  Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

✓ Training configuration set
  Total timesteps: 200,000
  Update frequency: 2048
  Using device: cuda


## 6. Train the Agent

In [7]:
# Create environment and agent
env = ConstrainedAircraftEnv()
agent = PPOLagrangian(
    env=env,
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_epsilon=0.2,
    c1=0.5,
    c2=0.01,
    cost_limit=0.0,
    lagrange_lr=0.01,
    hidden_dim=256
)

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

# Training loop
obs, _ = env.reset()
episode_reward = 0
episode_cost = 0
episode_length = 0
episode_violations = 0
start_time = time.time()

for timestep in range(TOTAL_TIMESTEPS):
    action, log_prob, value, cost_value = agent.select_action(obs)
    action = np.clip(action, env.action_space.low, env.action_space.high)

    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    cost = info['constraint_cost']
    agent.store_transition(obs, action, reward, cost, value, cost_value, log_prob, done)

    episode_reward += reward
    episode_cost += cost
    episode_length += 1
    if not info['is_safe']:
        episode_violations += 1

    obs = next_obs

    if done:
        agent.episode_rewards.append(episode_reward)
        agent.episode_costs.append(episode_cost)
        agent.episode_lengths.append(episode_length)
        agent.constraint_violations.append(episode_violations)

        obs, _ = env.reset()
        episode_reward = 0
        episode_cost = 0
        episode_length = 0
        episode_violations = 0

    if len(agent.buffer['obs']) >= UPDATE_FREQ:
        update_info = agent.update()

        if len(agent.episode_rewards) > 0 and len(agent.episode_rewards) % LOG_FREQ == 0:
            clear_output(wait=True)
            elapsed = time.time() - start_time
            print(f"\n{'='*60}")
            print(f"Timestep: {timestep:,} / {TOTAL_TIMESTEPS:,} ({100*timestep/TOTAL_TIMESTEPS:.1f}%)")
            print(f"Elapsed Time: {elapsed/60:.1f} minutes")
            print(f"{'='*60}")
            print(f"Avg Episode Reward: {np.mean(agent.episode_rewards):.2f}")
            print(f"Avg Episode Cost: {np.mean(agent.episode_costs):.4f}")
            print(f"Avg Episode Length: {np.mean(agent.episode_lengths):.1f}")
            print(f"Avg Violations/Episode: {np.mean(agent.constraint_violations):.2f}")
            print(f"Lambda (Lagrange): {update_info['lambda']:.4f}")
            print(f"Policy Loss: {update_info['policy_loss']:.4f}")
            print(f"Value Loss: {update_info['value_loss']:.4f}")
            print(f"{'='*60}")

print("\n✓ Training completed!")

/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(



STARTING TRAINING


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

## 7. Plot Training Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Episode rewards
axes[0, 0].plot(agent.episode_rewards)
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Episode Reward')
axes[0, 0].set_title('Training Rewards')
axes[0, 0].grid(True)

# Episode costs
axes[0, 1].plot(agent.episode_costs)
axes[0, 1].axhline(y=agent.cost_limit, color='r', linestyle='--', label='Cost Limit')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Episode Cost')
axes[0, 1].set_title('Constraint Violations')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Episode lengths
axes[1, 0].plot(agent.episode_lengths)
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Episode Length')
axes[1, 0].set_title('Episode Lengths')
axes[1, 0].grid(True)

# Violations per episode
axes[1, 1].plot(agent.constraint_violations)
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Violations per Episode')
axes[1, 1].set_title('Safety Violations per Episode')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print("✓ Training curves plotted")

## 8. Evaluate Trained Policy

In [ ]:
print("\n" + "="*60)
print("EVALUATING TRAINED POLICY")
print("="*60)

n_eval_episodes = 5
eval_rewards = []
eval_costs = []
success_count = 0

# Store trajectory of first episode
trajectory = {
    'time': [], 'heading': [], 'altitude': [], 'velocity': [],
    'load_factor': [], 'roll_angle': [], 'pitch_angle': []
}

for episode in range(n_eval_episodes):
    obs, _ = env.reset()
    episode_reward = 0
    episode_cost = 0
    done = False
    step = 0

    while not done:
        action, _, _, _ = agent.select_action(obs, deterministic=True)
        action = np.clip(action, env.action_space.low, env.action_space.high)

        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        episode_reward += reward
        episode_cost += info['constraint_cost']

        # Record first episode trajectory
        if episode == 0:
            trajectory['time'].append(step * env.dt)
            trajectory['heading'].append(np.rad2deg(env.state[8]))
            trajectory['altitude'].append(-env.state[11])
            trajectory['velocity'].append(info['velocity'])
            trajectory['load_factor'].append(info['load_factor'])
            trajectory['roll_angle'].append(np.rad2deg(env.state[6]))
            trajectory['pitch_angle'].append(np.rad2deg(env.state[7]))

        step += 1

    eval_rewards.append(episode_reward)
    eval_costs.append(episode_cost)

    heading_error = info['heading_error']
    altitude_error = info['altitude_error']

    if heading_error < np.deg2rad(5) and altitude_error < env.altitude_tolerance:
        success_count += 1
        status = "SUCCESS"
    else:
        status = "FAILED"

    print(f"Episode {episode+1}: {status} - Reward: {episode_reward:.2f}, Cost: {episode_cost:.4f}")
    if status == "FAILED":
        print(f"  Heading error: {np.rad2deg(heading_error):.2f}°, Altitude error: {altitude_error:.2f}m")

print("\n" + "="*60)
print(f"Success Rate: {success_count}/{n_eval_episodes} ({100*success_count/n_eval_episodes:.1f}%)")
print(f"Average Reward: {np.mean(eval_rewards):.2f} ± {np.std(eval_rewards):.2f}")
print(f"Average Cost: {np.mean(eval_costs):.4f} ± {np.std(eval_costs):.4f}")
print("="*60)

## 9. Visualize Trajectory

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Heading
axes[0, 0].plot(trajectory['time'], trajectory['heading'])
axes[0, 0].axhline(y=180, color='r', linestyle='--', label='Target')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Heading (degrees)')
axes[0, 0].set_title('Heading Angle')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Altitude
axes[0, 1].plot(trajectory['time'], trajectory['altitude'])
axes[0, 1].axhline(y=3000, color='r', linestyle='--', label='Initial')
axes[0, 1].fill_between(trajectory['time'], 2950, 3050, alpha=0.3, color='green', label='Tolerance')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Altitude (m)')
axes[0, 1].set_title('Altitude')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Velocity
axes[1, 0].plot(trajectory['time'], trajectory['velocity'])
axes[1, 0].axhline(y=200, color='r', linestyle='--', label='Target')
axes[1, 0].axhline(y=300, color='orange', linestyle=':', label='Max limit')
axes[1, 0].axhline(y=100, color='orange', linestyle=':', label='Min limit')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Velocity (m/s)')
axes[1, 0].set_title('Velocity')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Load factor
axes[1, 1].plot(trajectory['time'], trajectory['load_factor'])
axes[1, 1].axhline(y=7.0, color='r', linestyle='--', label='Max limit (+7g)')
axes[1, 1].axhline(y=-3.0, color='r', linestyle='--', label='Min limit (-3g)')
axes[1, 1].fill_between(trajectory['time'], -3, 7, alpha=0.2, color='green')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Load Factor (g)')
axes[1, 1].set_title('Load Factor')
axes[1, 1].legend()
axes[1, 1].grid(True)

# Roll angle
axes[2, 0].plot(trajectory['time'], trajectory['roll_angle'])
axes[2, 0].axhline(y=85, color='r', linestyle='--', label='Max limit')
axes[2, 0].axhline(y=-85, color='r', linestyle='--', label='Min limit')
axes[2, 0].set_xlabel('Time (s)')
axes[2, 0].set_ylabel('Roll Angle (degrees)')
axes[2, 0].set_title('Roll Angle')
axes[2, 0].legend()
axes[2, 0].grid(True)

# Pitch angle
axes[2, 1].plot(trajectory['time'], trajectory['pitch_angle'])
axes[2, 1].axhline(y=60, color='r', linestyle='--', label='Max limit')
axes[2, 1].axhline(y=-60, color='r', linestyle='--', label='Min limit')
axes[2, 1].set_xlabel('Time (s)')
axes[2, 1].set_ylabel('Pitch Angle (degrees)')
axes[2, 1].set_title('Pitch Angle')
axes[2, 1].legend()
axes[2, 1].grid(True)

plt.tight_layout()
plt.show()

print("✓ Trajectory visualization complete")

## 10. Save Model (Optional)

In [ ]:
# Save the trained model
model_path = 'ppo_lagrangian_aircraft.pth'
torch.save({
    'policy_state_dict': agent.policy.state_dict(),
    'optimizer_state_dict': agent.optimizer.state_dict(),
    'lambda': agent.lambda_net.item()
}, model_path)

print(f"✓ Model saved to {model_path}")
print("\nTo download the model:")
print("  from google.colab import files")
print("  files.download('ppo_lagrangian_aircraft.pth')")